# Assignment 2 - Build a Convolutional Neural Network using TensorFlow or PyTorch for image classification on the CIFAR-10 or MNIST dataset, visualize convolution and pooling outputs across layers, experiment with various kernel sizes, layer depths, and activation functions, and evaluate model performance using accuracy, precision, recall, and a confusion matrix.


## 1. Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

## 2. Load CIFAR-10 Dataset

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=64, shuffle=False)

print(f"Training images: {len(trainset)}")
print(f"Test images: {len(testset)}")

## 3. Visualize Sample Images

In [ ]:
dataiter = iter(trainloader)
images, labels = next(dataiter)
images_show = images[:8]
labels_show = labels[:8]

fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i in range(8):
    img = images_show[i] * 0.5 + 0.5
    img = img.permute(1, 2, 0).numpy()
    axes[i].imshow(img)
    axes[i].set_title(classes[labels_show[i]])
    axes[i].axis('off')
plt.suptitle("Sample CIFAR-10 Images")
plt.tight_layout()
plt.show()

## 4. Define CNN Model

In [ ]:
class MyCNN(nn.Module):
    def __init__(self, kernel_size=3, num_filters=32, activation='relu'):
        super(MyCNN, self).__init__()
        self.activation_name = activation

        self.conv1 = nn.Conv2d(3, num_filters, kernel_size=kernel_size, padding=1)
        self.conv2 = nn.Conv2d(num_filters, num_filters, kernel_size=kernel_size, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)

        self.conv3 = nn.Conv2d(num_filters, num_filters*2, kernel_size=kernel_size, padding=1)
        self.conv4 = nn.Conv2d(num_filters*2, num_filters*2, kernel_size=kernel_size, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)

        self.conv5 = nn.Conv2d(num_filters*2, num_filters*4, kernel_size=kernel_size, padding=1)
        self.pool3 = nn.MaxPool2d(2, 2)

        flatten_size = num_filters * 4 * 4 * 4

        self.fc1     = nn.Linear(flatten_size, 256)
        self.dropout = nn.Dropout(0.4)
        self.fc2     = nn.Linear(256, 10)

    def get_activation(self, x):
        if self.activation_name == 'relu':
            return torch.relu(x)
        elif self.activation_name == 'tanh':
            return torch.tanh(x)
        elif self.activation_name == 'sigmoid':
            return torch.sigmoid(x)

    def forward(self, x):
        x = self.get_activation(self.conv1(x))
        x = self.get_activation(self.conv2(x))
        x = self.pool1(x)

        x = self.get_activation(self.conv3(x))
        x = self.get_activation(self.conv4(x))
        x = self.pool2(x)

        x = self.get_activation(self.conv5(x))
        x = self.pool3(x)

        x = x.view(x.size(0), -1)
        x = self.get_activation(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# quick check
model_test = MyCNN()
dummy = torch.randn(2, 3, 32, 32)
out = model_test(dummy)
print("Output shape:", out.shape)  # should be [2, 10]

## 5. Visualize Conv and Pooling Outputs

In [ ]:
def visualize_conv_outputs(model, image_tensor):
    model.eval()
    with torch.no_grad():
        img = image_tensor.unsqueeze(0).to(device)
        out_conv1 = torch.relu(model.conv1(img))
        out_pool1 = model.pool1(torch.relu(model.conv2(out_conv1)))

    fig, axes = plt.subplots(2, 8, figsize=(16, 4))
    for i in range(8):
        axes[0][i].imshow(out_conv1[0][i].cpu().numpy(), cmap='viridis')
        axes[0][i].axis('off')
        axes[0][i].set_title(f'f{i+1}', fontsize=8)
        axes[1][i].imshow(out_pool1[0][i].cpu().numpy(), cmap='viridis')
        axes[1][i].axis('off')
        axes[1][i].set_title(f'f{i+1}', fontsize=8)

    axes[0][0].set_ylabel("Conv Output", fontsize=9)
    axes[1][0].set_ylabel("Pool Output", fontsize=9)
    plt.suptitle("Conv and Pooling Layer Outputs")
    plt.tight_layout()
    plt.show()

# test on a sample image (using untrained model, just to show the visualization)
sample_image, _ = testset[5]
temp_model = MyCNN().to(device)
visualize_conv_outputs(temp_model, sample_image)

## 6. Training and Evaluation Functions

In [ ]:
def train_model(model, trainloader, testloader, epochs=10):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    train_losses = []
    test_accuracies = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        epoch_loss = running_loss / len(trainloader)
        train_acc  = 100 * correct / total
        test_acc   = evaluate(model, testloader)

        train_losses.append(epoch_loss)
        test_accuracies.append(test_acc)
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {epoch_loss:.3f} | Train Acc: {train_acc:.1f}% | Test Acc: {test_acc:.1f}%")

    return train_losses, test_accuracies


def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total  += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return 100 * correct / total

## 7. Train Main Model

In [ ]:
print("Training main CNN (kernel=3, filters=32, relu)...")
main_model = MyCNN(kernel_size=3, num_filters=32, activation='relu')
train_losses, test_accs = train_model(main_model, trainloader, testloader, epochs=10)

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(test_accs, color='orange')
plt.title("Test Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.grid(True)

plt.tight_layout()
plt.show()

## 8. Visualize Trained Model Conv Outputs

In [ ]:
sample_image, sample_label = testset[5]
print("Sample image class:", classes[sample_label])
visualize_conv_outputs(main_model, sample_image)

## 9. Experiment - Different Kernel Sizes

In [ ]:
kernel_results = {}
for ks in [3, 5]:
    print(f"\nKernel size = {ks}")
    m = MyCNN(kernel_size=ks, num_filters=32, activation='relu')
    _, accs = train_model(m, trainloader, testloader, epochs=5)
    kernel_results[ks] = accs
    print(f"  Final Test Acc: {accs[-1]:.2f}%")

plt.figure(figsize=(6, 4))
for ks, accs in kernel_results.items():
    plt.plot(accs, label=f"kernel={ks}")
plt.title("Kernel Size Comparison")
plt.xlabel("Epoch")
plt.ylabel("Test Accuracy (%)")
plt.legend()
plt.grid(True)
plt.show()

## 10. Experiment - Different Activation Functions

In [ ]:
act_results = {}
for act in ['relu', 'tanh']:
    print(f"\nActivation = {act}")
    m = MyCNN(kernel_size=3, num_filters=32, activation=act)
    _, accs = train_model(m, trainloader, testloader, epochs=5)
    act_results[act] = accs
    print(f"  Final Test Acc: {accs[-1]:.2f}%")

plt.figure(figsize=(6, 4))
for act, accs in act_results.items():
    plt.plot(accs, label=act)
plt.title("Activation Function Comparison")
plt.xlabel("Epoch")
plt.ylabel("Test Accuracy (%)")
plt.legend()
plt.grid(True)
plt.show()

## 11. Confusion Matrix and Classification Report

In [ ]:
main_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        outputs = main_model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title("Confusion Matrix - CIFAR-10")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.show()

In [ ]:
print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=classes))

final_acc = evaluate(main_model, testloader)
print(f"\nFinal Test Accuracy: {final_acc:.2f}%")